In [0]:
"""
Finding User Purchases

Identify returning active users by finding users who made a second purchase within 1 to 7 days after their first purchase. Ignore same-day purchases. Output a list of these user_ids.

"""

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date, lag, datediff
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("UserPurchases").getOrCreate()

# Dummy data: id, user_id, item, created_at, revenue
data = [
    (1, 101, "laptop", "2020-01-05", 1000),
    (2, 101, "mouse", "2020-01-10", 25),     # 5 days after first → include
    (3, 102, "keyboard", "2020-01-15", 50),
    (4, 102, "monitor", "2020-01-16", 200),  # 1 day after first → include
    (5, 103, "phone", "2020-02-01", 800),
    (6, 103, "case", "2020-02-01", 20),      # same day → ignore
    (7, 103, "charger", "2020-02-10", 30),   # 9 days after first → exclude (>7)
    (8, 104, "tablet", "2020-03-01", 400),
    (9, 104, "pen", "2020-03-15", 5),        # 14 days after → exclude
    (10, 105, "book", "2020-04-01", 15),
    (11, 105, "notebook", "2020-04-05", 8),  # 4 days after → include
    (12, 106, "desk", "2020-05-01", 300),    # only one purchase → exclude
]

# Create DataFrame
df = spark.createDataFrame(data, ["id", "user_id", "item", "created_at", "revenue"])
df = df.withColumn("created_at", to_date(col("created_at")))

df.createOrReplaceTempView("amazon_transactions")

# Preview
df.show()

In [0]:
spark.sql("""
SELECT DISTINCT user_id
FROM (
    SELECT 
        user_id,
        created_at,
        LEAD(created_at) OVER(PARTITION BY user_id ORDER BY created_at) AS next_purchase_date
    FROM amazon_transactions
) t
WHERE DATEDIFF(next_purchase_date, created_at) BETWEEN 1 AND 7
""").display()